# YOLOv8 Drone Detection Fine-tuning

Complete notebook for fine-tuning a pre-trained YOLOv8 model on a new drone dataset.

## Dataset Format
- Images: JPG files in train/val/test folders
- Labels: YOLO format (class_id x_center y_center width height - normalized)
- Class: 0 = drone

## 1. Install Required Packages

In [2]:
# Install Ultralytics YOLO
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.4 MB/s eta 0:00:00


In [3]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU")

CUDA available: False


## 2. Mount Google Drive

We need access to your Drive to get the dataset and save the results.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Prepare Data

Copying data from Drive to the local Colab VM (`/content`) is much faster for training than reading directly from Drive.

**Note:** This notebook fine-tunes a pre-trained model on a new dataset. Make sure you have a trained model saved in your Drive from a previous training run.

import os
import shutil
from pathlib import Path

# Define path for the new dataset in Drive
new_dataset_zip = '/content/drive/MyDrive/yolov8_project/new_dataset.zip'
new_dataset_root = '/content/new_dataset'

if not os.path.exists(new_dataset_zip):
    print(f"Error: Could not find {new_dataset_zip}. Please make sure you uploaded new_dataset.zip to the 'yolov8_project' folder in Drive.")
else:
    print("Found new dataset zip.")

    # Unzip
    if os.path.exists(new_dataset_root):
        shutil.rmtree(new_dataset_root)
    os.makedirs(new_dataset_root, exist_ok=True)

    print("Unzipping...")
    !unzip -q "{new_dataset_zip}" -d "{new_dataset_root}"
    print("Unzip complete.")


## 4. Import Libraries and Setup

Now we'll import all the necessary libraries for fine-tuning.


In [ ]:
import os
import shutil
import yaml
from pathlib import Path
from ultralytics import YOLO
import torch

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 5. Configuration

Configure your dataset paths and fine-tuning parameters here. Adjust these values based on your needs.


In [ ]:
# ========== CONFIGURATION ==========
# Dataset paths
drive_project_path = '/content/drive/MyDrive/yolov8_project'
new_dataset_root = '/content/new_dataset'

# Pre-trained model path (from previous training run)
PRETRAINED_MODEL_PATH = f'{drive_project_path}/runs/detect/train_drone_colab/weights/best.pt'

# Fine-tuning parameters
NUM_EPOCHS = 50              # Number of fine-tuning epochs
IMAGE_SIZE = 640             # Input image size
BATCH_SIZE = 16              # Batch size
NEW_DATA_YAML_PATH = '/content/new_data.yaml'

# Project settings
PROJECT_DIR = '/content/runs/detect'
RUN_NAME = 'finetune_drone_colab'

# Verify GPU is available
print("Configuration loaded!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: Running on CPU. Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU")
print(f"\nFine-tuning settings:")
print(f"  Pre-trained model: {PRETRAINED_MODEL_PATH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")


In [ ]:
## 6. Create Configuration File

We need to create a `new_data.yaml` that points to the local `/content/new_dataset` locations.
# Sometimes it unzips as /content/new_dataset/drone_dataset_split/... or just /content/new_dataset/...
# We'll look for the 'images' folder.
base_path = Path(new_dataset_root)
found_images = list(base_path.rglob('images'))

if not found_images:
    print("Error: Could not find 'images' folder after unzip.")
else:
    # Take the parent of the first 'images' folder found as the data root
    new_data_root = found_images[0].parent
    print(f"New data root located at: {new_data_root}")

    # Create new_data.yaml
    new_data_yaml = {
        'path': str(new_data_root),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {0: 'drone'}  # Assuming same class mapping; update if different
    }

    with open(NEW_DATA_YAML_PATH, 'w') as f:
        yaml.dump(new_data_yaml, f)

    print("Created /content/new_data.yaml")
    !cat {NEW_DATA_YAML_PATH}


## 7. Load Pre-trained Model

Now we'll load the pre-trained YOLOv8 model from your previous training run.


In [ ]:
# Load the best model from previous training run
print(f"Loading pre-trained model from: {PRETRAINED_MODEL_PATH}")

if os.path.exists(PRETRAINED_MODEL_PATH):
    model = YOLO(PRETRAINED_MODEL_PATH)
    print("Model loaded successfully!")
else:
    print(f"Error: Model not found at {PRETRAINED_MODEL_PATH}")
    print("Please ensure you have completed training and saved the model to Drive.")
    print("Checking available weights...")
    if os.path.exists(f'{drive_project_path}/runs/detect'):
        print(os.listdir(f'{drive_project_path}/runs/detect'))


## 8. Fine-tune Model

This is where the magic happens! The model will learn from the new dataset while leveraging knowledge from the pre-trained model. Fine-tuning may take a while depending on your dataset size and number of epochs.

**Note:** The best model (lowest validation loss) will be saved automatically during fine-tuning.


In [ ]:
print("=" * 60)
print("Starting fine-tuning...")
print(f"Fine-tuning for {NUM_EPOCHS} epochs")
print(f"Device: {device}")
print(f"Pre-trained model: {PRETRAINED_MODEL_PATH}")
print("=" * 60)

# Fine-tune
results = model.train(
    data=NEW_DATA_YAML_PATH,
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True
)


## 9. View Training History

Let's visualize how the fine-tuning progressed over time. YOLOv8 automatically generates training plots that are saved in the runs directory.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

# Display training results
results_dir = f'{PROJECT_DIR}/{RUN_NAME}'
results_image = f'{results_dir}/results.png'

if os.path.exists(results_image):
    print("Fine-tuning results visualization:")
    display(Image(results_image))
else:
    print(f"Results image not found at {results_image}")
    print("Available files in results directory:")
    if os.path.exists(results_dir):
        print(glob.glob(f'{results_dir}/*'))


## 10. Evaluate Model

Let's evaluate the fine-tuned model on the test dataset to see how well it performs.


In [ ]:
# Load your best fine-tuned model from the fine-tuning results
model_path = f'{results_dir}/weights/best.pt'
if os.path.exists(model_path):
    print(f"Loading best model from: {model_path}")
    model = YOLO(model_path)
    
    # Run validation on the test split defined in new_data.yaml
    print("\nEvaluating on test dataset...")
    metrics = model.val(data=NEW_DATA_YAML_PATH, split='test')
    
    print(f"\nEvaluation Results:")
    print(f"  mAP50-95: {metrics.box.map:.4f}")
    print(f"  mAP50: {metrics.box.map50:.4f}")
else:
    print(f"Model not found at {model_path}")
    print("Please ensure fine-tuning completed successfully.")


## 11. Test Model on Validation Images

Let's see how well the model performs by visualizing predictions on some validation/test images.


In [ ]:
import random
import glob
from PIL import Image
from IPython.display import display

# Path to your test images based on your new dataset root
# Use new_data_root from the configuration file creation step
test_images_path = f'{new_data_root}/images/test/*.jpg'
test_images = glob.glob(test_images_path)

if not test_images:
    print("No images found in the test folder.")
else:
    # Select a random image
    random_image = random.choice(test_images)

    # Run prediction
    print(f"Predicting on: {random_image}")
    results = model.predict(source=random_image, conf=0.25)

    # Visualize the results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = Image.fromarray(im_array[..., ::-1])  # convert to RGB PIL image
        display(im)


In [ ]:
## 12. Test Model on Uploaded Images

You can test the model on your own images by uploading them.


In [ ]:
from google.colab import files

# Ensure model is loaded
if 'model' not in locals() or model is None:
    model_path = f'{results_dir}/weights/best.pt'
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"Loaded model from: {model_path}")
    else:
        print("Error: Model not found. Please run fine-tuning first.")

# Upload your image
print("Please select an image to upload:")
uploaded = files.upload()

# Process the uploaded file(s)
for filename in uploaded.keys():
    print(f'\nRunning prediction on: {filename}')

    # Run prediction
    results = model.predict(source=filename, conf=0.25)

    # Visualize and display the results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = Image.fromarray(im_array[..., ::-1])  # convert to RGB PIL image
        display(im)


In [ ]:
## 13. Save Results to Drive

Compress the fine-tuning results and copy them back to Google Drive for safekeeping.


In [ ]:
# Compress the fine-tuning results
zip_path = f'{results_dir}/runs_finetune.zip'
!zip -r {zip_path} {results_dir}

# Copy to Drive
destination = f'{drive_project_path}/runs_finetune.zip'
shutil.copy2(zip_path, destination)

print(f"✓ Results saved to Google Drive: {destination}")
print("\n🎉 ALL DONE! Your fine-tuned model is saved in Google Drive.")
